# Trend Agent

Looks through the SAP article chunks and pulls out emerging trends
(technology trends, customer behaviour shifts, industry developments).

Same setup as the other two agents — FAISS index from `embedding_v2.ipynb`,
local Ollama model for reasoning.


In [1]:
import json
import os
import time

import faiss
import pandas as pd
import requests
from sentence_transformers import SentenceTransformer
from pathlib import Path

cwd = Path.cwd()
DATA_DIR = next((p for p in [cwd / "notebook" / "data", cwd.parent / "notebook" / "data"] if p.exists()), cwd)

CHUNKS_PATH = DATA_DIR / "chunked_data.json"
INDEX_PATH  = DATA_DIR / "sap_intelligence.index"
OUT_PATH    = DATA_DIR / "trends.json"

EMBED_MODEL  = "BAAI/bge-small-en-v1.5"
OLLAMA_HOST  = os.getenv("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "phi4-mini")
TIMEOUT      = 300

print("data dir:", DATA_DIR)


data dir: /workspaces/AI-CEO-Strategic-Intelligence-Agent/notebook/data


In [2]:
chunks_df = pd.read_json(CHUNKS_PATH)
index = faiss.read_index(str(INDEX_PATH))
model = SentenceTransformer(EMBED_MODEL)

print("chunks loaded:", len(chunks_df))
print("vectors in index:", index.ntotal)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

chunks loaded: 1387
vectors in index: 1387


In [3]:
def retrieve(query, k=8):
    q_vec = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q_vec, k)
    results = chunks_df.iloc[idx[0]]
    return results["text"].tolist()


def ask_llm(prompt):
    resp = requests.post(
        f"{OLLAMA_HOST}/api/generate",
        json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False},
        timeout=TIMEOUT,
    )
    resp.raise_for_status()
    return resp.json()["response"]


def warmup():
    print("warming up model...")
    start = time.time()
    ask_llm("hi")
    print("warmup done in", round(time.time() - start, 1), "sec")


warmup()


warming up model...
warmup done in 1.9 sec


## Build context

In [4]:
queries = [
    "SAP technology trends and innovation",
    "SAP customer behaviour and adoption",
    "SAP industry developments",
]

chunks = []
for q in queries:
    chunks.extend(retrieve(q, k=6))

chunks = list(dict.fromkeys(chunks))

context = "\n\n".join(chunks)
print("number of chunks used:", len(chunks))
print("context length (chars):", len(context))


number of chunks used: 17
context length (chars): 15854


## Trend agent

Returns a JSON list with fields: `title`, `category`, `description`, `evidence`.
These feed the "emerging technologies" part of the Market Intelligence
dashboard section.


In [5]:
def trend_agent(context):
    prompt = f"""You are a market trend analyst for SAP.

Read the news context below and identify 3 to 5 emerging trends.
Trends can be technology trends, customer behaviour shifts, or industry developments.

Return ONLY a JSON array, no extra text, no markdown. Each item must look like this:

{{
  "title": "short trend title",
  "category": "Technology, Customer Behaviour, or Industry",
  "description": "1-2 sentences explaining the trend",
  "evidence": "one sentence quoting or summarising the supporting news"
}}

Context:
{context}
"""
    raw = ask_llm(prompt)
    return raw


raw_output = trend_agent(context)
print(raw_output)


[
    {
      "title": "AI-powered customer experiences",
      "category": "Technology",
      "description": "SAP's Advanced Success Plan embeds AI adoption patterns directly into delivery approaches, prioritizing high-impact use cases and guiding model governance.",
      "evidence": "The results are prioritized engagement with measurable outcomes like improved loyalty."
    },
    {
      "title": "Unified customer experience orchestration across channels",
      "category": "Technology",
      "description": "SAP Engagement Cloud enables brands to activate journeys triggered by real-time events, breaking down silos and enabling enterprise-scale execution.",
      "evidence": "Customers can expect effortless experiences with personalized interactions."
    },
    {
      "title": "Supply chain transformation towards autonomy",
      "category": "Industry Development",
      "description": "SAP Sapphire announced innovations like SAP Cloud ERP Private to accelerate the Autonomous En

## Clean up the JSON

In [6]:
def parse_json_list(raw_text):
    text = raw_text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        text = text.replace("json", "", 1).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        print("could not parse JSON, here is the raw text:")
        print(text)
        return []


trends = parse_json_list(raw_output)
print("parsed", len(trends), "trends")
for t in trends:
    print("-", t.get("title"), "|", t.get("category"))


parsed 5 trends
- AI-powered customer experiences | Technology
- Unified customer experience orchestration across channels | Technology
- Supply chain transformation towards autonomy | Industry Development
- Enhanced field services with integration across operational domains | Technology
- Continuous improvement with integrated software updates | Technology


## Save results for the dashboard

In [7]:
with open(OUT_PATH, "w") as f:
    json.dump(trends, f, indent=2)

print("saved to", OUT_PATH)


saved to /workspaces/AI-CEO-Strategic-Intelligence-Agent/notebook/data/trends.json
